# Lab 02 — VoltLens: classificador por transferência *legível / ilegível / não é medidor*
Bloco profundo (parte B): MobileNetV3-Small pré-treinada + cabeça de 3 classes **exclusivas** (softmax + entropia cruzada).
As fotos do cliente **não** estão no repositório: ajuste `FOTOS_DIR` abaixo.

In [ ]:
# ---------- CONFIGURAÇÃO (única célula a editar) ----------
from pathlib import Path
FOTOS_DIR = Path(r"CAMINHO/PARA/AS/FOTOS")      # <FOTOS_DIR>/<lote>/<nome_arquivo>.jpg  (NÃO versionar)
LOTES = {"treino": ["20250520", "20250521"],   # nomes das subpastas de lote (ajustar aos reais)
         "val":    ["20250522"],
         "teste":  ["20250703"]}
RAIZ = Path.cwd() if (Path.cwd() / "rotulos").exists() else Path("lab-02")
ROTULOS_CSV = RAIZ / "rotulos" / "rotulos.csv"
MAPA_IDS_CSV = RAIZ / "rotulos" / "ids.csv"      # nome_arquivo, id_leitura, numero_medidor (extraído dos CSVs do cliente; também NÃO versionar se identificar cliente)
CLASSES = ["legivel", "ilegivel", "nao_medidor"]
SEMENTES = [0, 1, 2]
ALTURA, LARGURA = 480, 360

In [ ]:
import math, time, warnings
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt
torch.set_num_threads(2); torch.manual_seed(42); np.random.seed(42)
MEDIA, DP = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1), torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

## C — Rótulos e partição por lote

In [ ]:
df = pd.read_csv(ROTULOS_CSV)
lote_para_conj = {l: c for c, ls in LOTES.items() for l in ls}
df["conjunto"] = df["lote"].astype(str).map(lote_para_conj)
assert df["conjunto"].notna().all(), "lote fora de LOTES: ajuste a configuração"
print(len(df), "fotos rotuladas")
print(pd.crosstab(df["lote"], df["rotulo"], margins=True))
print(pd.crosstab(df["conjunto"], df["rotulo"], margins=True))

In [ ]:
# C4 — prova numérica de ausência de vazamento (partição por lote, nunca por foto)
ids = pd.read_csv(MAPA_IDS_CSV)                      # id_leitura = nome sem sufixo _00N; numero_medidor dos CSVs do cliente
d = df.merge(ids, on="nome_arquivo", how="left")
assert d["id_leitura"].notna().all() and d["numero_medidor"].notna().all(), "ids.csv incompleto"
conj = {c: d[d.conjunto == c] for c in ["treino", "val", "teste"]}
for a, b in [("treino","val"), ("treino","teste"), ("val","teste")]:
    il = set(conj[a].id_leitura) & set(conj[b].id_leitura)
    nm = set(conj[a].numero_medidor) & set(conj[b].numero_medidor)
    print(f"{a:>6} x {b:<6} | leituras em comum: {len(il)} | medidores em comum: {len(nm)}")
    assert not il, "vazamento por leitura"
    if nm: warnings.warn(f"{len(nm)} medidores em comum entre {a} e {b}: remover do conjunto menor")

## B — Modelo, tabela de formas e parâmetros

In [ ]:
class Classificador(nn.Module):
    def __init__(self, pretreinado=True, p_dropout=0.2):
        super().__init__()
        base = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretreinado else None)
        self.extrator = base.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.cabeca = nn.Sequential(nn.Dropout(p_dropout), nn.Linear(576, len(CLASSES)))
    def features(self, x): return self.pool(self.extrator(x)).flatten(1)
    def forward(self, x): return self.cabeca(self.features(x))   # LOGITS: sem softmax aqui

def tabela_de_formas(m, x):
    linhas = []
    hs = [mod.register_forward_hook(lambda _m, _i, o, n=n: linhas.append((n, tuple(o.shape))))
          for n, mod in [(f"extrator.{n}", c) for n, c in m.extrator.named_children()] + [("pool", m.pool), ("cabeca", m.cabeca)]]
    with torch.no_grad(): m.eval()(x)
    for h in hs: h.remove()
    print(f"{'camada':<16}{'saída':>24}")
    for n, s in linhas: print(f"{n:<16}{str(s):>24}")

def contar(m):
    return sum(p.numel() for p in m.parameters()), sum(p.numel() for p in m.parameters() if p.requires_grad)

modelo = Classificador()
tabela_de_formas(modelo, torch.randn(1, 3, ALTURA, LARGURA))
print("ajuste fino  : total %s | treináveis %s" % tuple(f"{v:,}" for v in contar(modelo)))
for p in modelo.extrator.parameters(): p.requires_grad = False
print("extração     : total %s | treináveis %s" % tuple(f"{v:,}" for v in contar(modelo)))

**B3 — ativação e perda.** As três classes são mutuamente exclusivas (uma foto é uma coisa só), portanto `softmax` + `CrossEntropyLoss`. O modelo devolve *logits*; a softmax só é aplicada na inferência (E). Exemplo real que prova a exclusividade: _preencher com o nome de um arquivo de `rotulos.csv` e a classe única atribuída_.

In [ ]:
def carregar(nome, lote):
    img = Image.open(FOTOS_DIR / str(lote) / nome).convert("RGB")
    assert img.size == (LARGURA, ALTURA), f"{nome}: {img.size}"
    t = torch.from_numpy(np.asarray(img).copy()).permute(2, 0, 1).float() / 255
    return (t - MEDIA) / DP
def tensores(sub):
    X = torch.stack([carregar(r.nome_arquivo, r.lote) for r in sub.itertuples()])
    y = torch.tensor([CLASSES.index(r) for r in sub.rotulo])
    return X, y
dados = {c: tensores(df[df.conjunto == c].reset_index(drop=True)) for c in ["treino", "val", "teste"]}
print({c: tuple(v[0].shape) for c, v in dados.items()})

## D — Sanidade

In [ ]:
# D1 — perda inicial com pesos aleatórios (cabeça aleatória, extrator pré-treinado) vs ln(3)
m0 = Classificador(); m0.eval()
Xv, yv = dados["val"]
with torch.no_grad(): ce0 = F.cross_entropy(m0(Xv[:32]), yv[:32]).item()
print(f"perda inicial = {ce0:.3f} | ln(3) = {math.log(3):.3f}")

In [ ]:
# D2 — sobreajustar 16 fotos reais (sem dropout, tudo treinável) até perda ~0
m2 = Classificador(); m2.cabeca[0].p = 0.0
Xt, yt = dados["treino"]; idx = torch.randperm(len(Xt))[:16]
xb, yb = Xt[idx], yt[idx]
opt = torch.optim.Adam(m2.parameters(), lr=1e-3); m2.train()
for i in range(80):
    opt.zero_grad(); l = F.cross_entropy(m2(xb), yb); l.backward(); opt.step()
    if i in (0, 79): print(f"passo {i:>2}: perda = {l.item():.4f}")
assert l.item() < 0.1, "não decorou 16 fotos: diagnosticar rede/perda/rótulos antes de seguir"

In [ ]:
# D3 — extração de características (cabeça treinada sobre extrator congelado) vs baselines, 3 sementes
base = Classificador(); base.eval()
@torch.no_grad()
def feats(X): return torch.cat([base.features(X[i:i+32]) for i in range(0, len(X), 32)])
Ftr, Fva, Fte = (feats(dados[c][0]) for c in ["treino", "val", "teste"])
ytr, yva, yte = (dados[c][1] for c in ["treino", "val", "teste"])

def treina_cabeca(seed, epocas=200):
    torch.manual_seed(seed)
    cab = nn.Linear(576, 3); opt = torch.optim.AdamW(cab.parameters(), lr=1e-2, weight_decay=1e-3)
    peso = (len(ytr) / (3 * torch.bincount(ytr, minlength=3).clamp(min=1))).float()
    for _ in range(epocas):
        opt.zero_grad(); F.cross_entropy(cab(Ftr), ytr, weight=peso).backward(); opt.step()
    return cab

def classico(X):   # baseline não profunda: histograma de cinza (16 bins) + variância do Laplaciano
    out = []
    for x in X:
        g = (x * DP + MEDIA).mean(0).numpy()
        lap = g[1:-1,1:-1]*4 - g[:-2,1:-1] - g[2:,1:-1] - g[1:-1,:-2] - g[1:-1,2:]
        out.append(np.r_[np.histogram(g, 16, (0, 1), density=True)[0], lap.var()])
    return np.array(out)
Ctr, Cva, Cte = (classico(dados[c][0]) for c in ["treino", "val", "teste"])

res = {"profundo (extração)": [], "LR sobre histograma+Laplaciano": []}
for s in SEMENTES:
    cab = treina_cabeca(s)
    with torch.no_grad(): p = cab(Fva).argmax(1).numpy()
    res["profundo (extração)"].append(f1_score(yva, p, average="macro"))
    lr = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=s).fit(Ctr, ytr)
    res["LR sobre histograma+Laplaciano"].append(f1_score(yva, lr.predict(Cva), average="macro"))
print("F1-macro na validação (média ± desvio, 3 sementes):")
for k, v in res.items(): print(f"  {k:<34} {np.mean(v):.3f} ± {np.std(v):.3f}   {np.round(v, 3)}")

_Leitura: se o profundo perder da baseline, explicar por quê no README (D3)._

## E — Confiança e recusa (ajustados na **validação**, avaliados no teste)

In [ ]:
cab = treina_cabeca(0)
with torch.no_grad(): Lva, Lte = cab(Fva), cab(Fte)

def ece(p, y, bins=10):
    conf, pred = p.max(1); acerto = (pred == y).float(); e = 0.0
    for lo in np.linspace(0, 1, bins, endpoint=False):
        m = (conf > lo) & (conf <= lo + 1 / bins)
        if m.any(): e += m.float().mean().item() * abs(acerto[m].mean().item() - conf[m].mean().item())
    return e

T = torch.ones(1, requires_grad=True)
opt = torch.optim.LBFGS([T], lr=0.1, max_iter=100)
def clos():
    opt.zero_grad(); l = F.cross_entropy(Lva / T, yva); l.backward(); return l
opt.step(clos); T = T.item()
print(f"T = {T:.3f}")
print(f"ECE validação: antes {ece(Lva.softmax(1), yva):.3f} | depois {ece((Lva/T).softmax(1), yva):.3f}")
print(f"ECE teste    : antes {ece(Lte.softmax(1), yte):.3f} | depois {ece((Lte/T).softmax(1), yte):.3f}")

fig, ax = plt.subplots(1, 2, figsize=(9, 4), sharey=True)
for a, (nome, L) in zip(ax, [("antes", Lva), ("depois (T)", Lva / T)]):
    p = L.softmax(1); conf, pred = p.max(1); ac = (pred == yva).float()
    b = np.linspace(0, 1, 6); xs, ys = [], []
    for lo, hi in zip(b[:-1], b[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): xs.append(conf[m].mean().item()); ys.append(ac[m].mean().item())
    a.plot([0, 1], [0, 1], "--", c="gray"); a.plot(xs, ys, "o-"); a.set_title(f"confiabilidade — {nome}"); a.set_xlabel("confiança")
ax[0].set_ylabel("acurácia"); plt.tight_layout(); plt.show()

In [ ]:
# E2 — cobertura x risco da decisão "legível" (limiar escolhido na validação; curva também no teste)
def curva(L, y, T):
    p = (L / T).softmax(1); conf, pred = p.max(1); ordem = conf.argsort(descending=True)
    err = (pred[ordem] != y[ordem]).float().cumsum(0) / torch.arange(1, len(y) + 1)
    return (torch.arange(1, len(y) + 1) / len(y)).numpy(), err.numpy()
cv, rv = curva(Lva, yva, T); ct, rt = curva(Lte, yte, T)
COBERTURA_ALVO, RISCO_ALVO = 0.50, 0.02      # meta: cobertura >= 50% com risco <= 2% (X = 98%); ver README V2
plt.plot(cv, rv, label="validação"); plt.plot(ct, rt, label="teste")
plt.scatter([COBERTURA_ALVO], [RISCO_ALVO], c="r", zorder=5, label="meta do kickoff")
plt.xlabel("cobertura"); plt.ylabel("risco (erro nas decididas)"); plt.legend(); plt.show()
i = np.searchsorted(cv, COBERTURA_ALVO)
print(f"risco em cobertura {COBERTURA_ALVO:.0%}: validação {rv[i]:.3f} | teste {rt[min(i, len(rt)-1)]:.3f} | alvo <= {RISCO_ALVO}")

## V — Parte específica (V3, V4)
Preencher com os CSVs do cliente (não versionados): V3 = regra de pareamento foto ↔ registro e contagem de órfãs; V4 = leituras com valor zero que têm foto, passadas pelo classificador. Ver `README.md`.

In [ ]:
# V4 (esqueleto): COLUNAS_ZERO = leituras de valor 0 com foto pareada -> classificador -> distribuição das classes previstas
# zeros = pd.read_csv(...)  # TODO: caminho dos CSVs do cliente